# Creating tables and views, and writing data

Tables store data; views store a query. In Unity Catalog, CREATE TABLE makes a managed Delta table by default. CREATE TABLE AS SELECT builds a table from a query, INSERT INTO appends rows, INSERT OVERWRITE and CREATE OR REPLACE replace them, and MERGE upserts by key. Every write is a new table version.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Creating tables and views, and writing data](https://www.datapreplab.com/#topic/sql-tables-writes) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** create a table, make the classic duplicate-append mistake, find it in history and fix it.

**Part A: create and append.**

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.orders_lab AS
SELECT * FROM VALUES (1, 100), (2, 50), (3, 75) AS t(order_id, amount);

CREATE OR REPLACE TEMP VIEW batch AS
SELECT * FROM VALUES (4, 20), (5, 30) AS t(order_id, amount);

INSERT INTO workspace.default.orders_lab SELECT * FROM batch;
SELECT count(*) FROM workspace.default.orders_lab;   -- 5

**Part B: the mistake.** Append the same batch again by accident, count, and look at the history:

In [ ]:
%sql
INSERT INTO workspace.default.orders_lab SELECT * FROM batch;
SELECT count(*) FROM workspace.default.orders_lab;   -- 7

In [ ]:
%sql
DESCRIBE HISTORY workspace.default.orders_lab;

You should see versions 0 (create), 1 (write) and 2 (write).

**Part C: fix and make it safe.** Restore version 1, then load the batch with MERGE twice and count after each run:

In [ ]:
%sql
RESTORE TABLE workspace.default.orders_lab TO VERSION AS OF 1;

MERGE INTO workspace.default.orders_lab AS t
USING batch AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
SELECT count(*) FROM workspace.default.orders_lab;   -- 5

Run the same MERGE again: the count stays 5.

In [ ]:
%sql
MERGE INTO workspace.default.orders_lab AS t
USING batch AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
SELECT count(*) FROM workspace.default.orders_lab;   -- 5

**Part D: a view.**

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.default.big_orders AS
SELECT * FROM workspace.default.orders_lab WHERE amount >= 50;
SELECT * FROM workspace.default.big_orders;

Expected evidence:

```text
Part A -> 5
Part B -> 7 ; history: version 0 CREATE OR REPLACE TABLE AS SELECT, versions 1 and 2 WRITE (mode Append)
Part C -> 5 after each MERGE ; RESTORE appears as a new version
Part D -> orders 1, 2 and 3
```

**Cleanup:**

In [ ]:
%sql
DROP VIEW IF EXISTS workspace.default.big_orders;
DROP TABLE IF EXISTS workspace.default.orders_lab;